# 36 — BẢN GHI CỬA SỔ 15 giây (văn bản → văn bản, cả hai batch)\n
Tầng này **không nhìn ảnh**. Nó đọc caption keyframe đã sinh rồi viết cho mỗi **cửa sổ 15 giây** một bản ghi mang đúng thứ caption keyframe KHÔNG thể có: **chuyện gì xảy ra**, **cái gì đổi**, và **cửa sổ này nối với cửa sổ trước thế nào**.\n
**Đặt `BATCH = 1` hoặc `BATCH = 2`** — cùng một notebook, cùng prompt, cùng đơn vị. Chỉ khác nguồn caption và nơi ghi.\n
**Vì sao CỬA SỔ chứ không phải SHOT:** batch 2 **không có ranh giới shot**. Đo trên `coordinate-provenance.jsonl` của chính các worker: `anchor_sec` chạy đều **2,0 s cho 7.880/7.880 cặp liên tiếp** (nhóm N) và 4,0 s (nhóm S) — lấy mẫu theo lưới thời gian, không phải theo cảnh. Dùng shot cho batch 1 và cửa sổ cho batch 2 là cho hai nửa kho hai đơn vị khác nhau, tức vẫn thiên lệch.\n
Cửa sổ còn thắng shot ở chỗ đo được: cửa sổ 15 giây **chứa trọn một cú chuyển cảnh**, nên `event` viết được *"introduces a story before cutting to footage of…"*. Shot 2,72 giây không bao giờ chứa nổi một chuyển cảnh. Và nó xoá luôn bài toán **10.089 shot mù** của batch 1 — cửa sổ chỉ gom khung CÓ THẬT.\n
**Vì sao text→text:** caption `v5_bao` có 49,7% ≥3 gốc động từ (Qwen cũ 5,0%), 185 token. ~669 token/đoạn thay vì ~4.485 nếu gọi lại VLM với ảnh.\n
⚠ **`pts_time` lấy từ `map-keyframes`, KHÔNG đoán fps.** Batch 1 có 4 giá trị fps (một video 26,44), batch 2 có 68. Đoán 25,0 là gom sai cửa sổ trên hàng chục nghìn khung mà không gì báo.\n
⚠ **Caption batch 2 là chuỗi THÔ hai khối** — `CAPTION_THO = True` bật `tach_caption_ocr`. Đưa nguyên chuỗi thô vào prompt là nhồi khối OCR hai lần.\n
## Trước khi Run All\n
1. Đặt `BATCH`, sửa `RUNNER`, đặt `RUN_SCOPE = "SHARD"` và `CONFIRM_SHARD` **bằng đúng** `RUNNER`.
2. Bật 4 secret: `TRAMMAY_API_KEY` · `GITHUB_TOKEN` · `MODAL_TOKEN_ID` · `MODAL_TOKEN_SECRET`.
3. **Không gắn dataset nào.** Caption và `map-keyframes` đều kéo từ Modal.\n

In [ ]:
from pathlib import Path

# ── ai chạy, chạy phạm vi nào ──────────────────────────────────────────────
RUNNER = "w1"                     # w1..w6
RUNNERS = ("w1", "w2", "w3", "w4", "w5", "w6")
RUN_SCOPE = "PREFLIGHT"           # PREFLIGHT | SHARD | AGGREGATE
CONFIRM_SHARD = ""                # SHARD: đặt đúng bằng RUNNER
CONFIRM_AGGREGATE = False         # AGGREGATE: True sau khi cả 6 shard COMPLETE

# ── cổng API ───────────────────────────────────────────────────────────────
# Kaggle Secrets: Add-ons -> Secrets -> TRAMMAY_API_KEY
TRAMMAY_SECRET = "TRAMMAY_API_KEY"
MODEL_NAME = "ag/gemini-3.8-flash"
PROMPT_VARIANT = "ban_ghi_cua_so"   # prompt ở mfe.offline.shot_record_prompt
# Cổng BỎ QUA `max_tokens` (đo 22-09: gửi 40 vẫn trả 263 token, `finish_reason
# ='stop'`), nên con số này chỉ để qua cửa kiểm tra. Trần của endpoint batch là
# 8.192; chat thường vẫn 4.096.
MAX_TOKENS = 8192
# ── đường gọi: "chat" (1 ảnh/1 request) hay "batch" (k ảnh/1 request) ───────
#
# Chọn theo thứ đang CHẶN, không theo thứ nghe nhanh hơn:
#
#   "batch"  khi trần REQUEST chặn (10.000/ngày/user). Giảm k lần số request,
#            đổi lại chậm hơn 1,47 lần và caption cần phụ lục `len180` để không
#            tụt từ 195 xuống 148 từ.
#   "chat"   khi trần request đã nới. Nhanh hơn (2.040 so 1.388 khung/giờ/
#            worker) và tái lập ĐÚNG cấu hình đã sinh 499 video đêm 21-09.
#
# ⚠ `BATCH_SIZE = 1` KHÔNG tương đương `"chat"`: nó vẫn đi qua
# `/chat/completions/batch`, tức vẫn dính system prompt ép JSON mà cổng chèn —
# chính thứ làm model tự chia phần. Muốn một-ảnh-một-request thật thì phải là
# `"chat"`.
DUONG_GOI = "chat"                # cụm cửa sổ LUÔN đi đường chat
MAX_TOKENS_CHAT = 8192            # ~8 bản ghi JSON trong một lượt
MOC_GHI = 80 if DUONG_GOI == "chat" else 10   # ghi checkpoint mỗi ~80 ảnh
BATCH_SIZE = 8                    # chỉ dùng khi DUONG_GOI == "batch"
BATCH_MAX_BYTES = 7_000_000       # hàng rào THẬT: cắt theo byte — xem bảng dưới
# Số luồng và nền lùi phải khớp ĐƯỜNG GỌI — cả hai đều theo độ dài một request:
#   "chat"  : ~8s/lượt  -> 28 luồng, lùi 2s   (đo đêm 21-09: 6–9 giờ sạch)
#   "batch" : 23–47s    ->  2 luồng, lùi 30s  (đo 22-09: 0/18 hỏng)
# Đặt lẫn là hỏng theo hai kiểu: 28 luồng batch thì 524 hàng loạt, còn 2 luồng
# chat thì chậm gấp 14 lần mà chẳng an toàn hơn.
MAX_WORKERS = 12                  # cụm nặng hơn một ảnh; 28 là quá tay
BACKOFF_BASE = 2.0 if DUONG_GOI == "chat" else 30.0
REQUEST_TIMEOUT = 300
MAX_RETRIES = 4

# Chạm hạn mức 5 tiếng của thượng nguồn thì NGỦ CHỜ, không thoát: phiên Kaggle
# sống 12 tiếng nên chờ hết một cửa sổ vẫn còn dư thời gian làm việc, còn thoát
# ra là phải có người ngồi canh bấm chạy lại (đo 24-09: một worker dừng lúc
# 4,95 giờ rồi nằm không suốt đêm).
#
# 330 phút = 5,5 tiếng: một cửa sổ đầy cộng biên. Chờ lâu hơn thì Kaggle giết
# phiên trước — mà chuyện đó vô hại, marker và checkpoint đã nằm trên volume.
CHO_QUOTA_PHUT = 330
# Thăm dò bằng một lượt gọi KHÔNG ẢNH. Không ai biết cửa sổ bắt đầu lúc nào, nên
# ngủ cứng 5 tiếng là luôn chọn trường hợp xấu nhất; thăm dò 10 phút một lần tốn
# tối đa ~30 request trên 1.000 của cửa sổ.
NHIP_THAM_DO_PHUT = 10

# ── đường dẫn ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/triet/structure"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/caption-gemini")

# Nguồn ảnh + map-keyframes: volume BTC, KHÔNG phải `/kaggle/input`.
BTC_VOLUME = KF_VOLUME
BTC_BUNDLE = KF_BUNDLE
CACHE_ANH = WORK_DIR / "keyframes"

# ── BATCH: 1 hay 2 ─────────────────────────────────────────────────────────
#
# Cùng một notebook cho cả hai nửa kho, vì đơn vị (cửa sổ 15 giây) và prompt
# giống hệt nhau. Chỉ khác NGUỒN caption và NƠI ghi kết quả.
#
# ⚠ Caption batch 2 là chuỗi THÔ hai khối `[FINAL_CAPTION]` + `[OCR]`, phải
# tách bằng `tach_caption_ocr` trước khi đưa vào prompt. Đưa nguyên chuỗi thô
# vào là nhồi khối OCR hai lần và làm loãng phần mô tả.
BATCH = 1                         # 1 | 2

CUA_SO_GIAY = 15.0                # §4.3: 3 mệnh đề ≈ 13,1s; đo thật 3,7 khung/cửa sổ
CUM_CAPTION_TOI_DA = 25           # ~25 caption/prompt ≈ 4.600 token
CUM_CUA_SO_TOI_DA = 8             # trần TRÊN; số caption mới là thứ quyết
MAX_KF_TRONG_CUA_SO = 10          # cửa sổ dày bất thường thì lấy mẫu thưa

if BATCH == 1:
    KF_VOLUME = "keyframes-btc"
    KF_BUNDLE = "aic_btc_b1_v1"
    CAPTION_VOLUME = "keyframe-augmentation-research"
    CAPTION_DIR = "/research/caption-gemini-trammay-v1/parts/captions"
    CAPTION_THO = False           # caption batch 1 đã là văn bản thuần
    OUTPUT_ROOT = "/research/window-records-v1"
    EXPECTED_VIDEOS = 873
else:
    KF_VOLUME = "keyframes-batch-2-v2"
    KF_BUNDLE = "aic_btc_b2_v1"
    CAPTION_VOLUME = "keyframes-batch-2-v2"
    CAPTION_DIR = "/bundle/aic_btc_b2_v1/parts/gemini_raw"
    CAPTION_THO = True            # phải tách [FINAL_CAPTION] khỏi [OCR]
    OUTPUT_ROOT = "/bundle/aic_btc_b2_v1"
    # ĐẾM 24-09 (chiều): 611 = M 304 · N 296 · S 11. Buổi sáng còn là 581 —
    # Trọng trích thêm 30 video nhóm M giữa hai lần đếm, nên ĐẾM LẠI trước mỗi
    # lượt chạy thay vì tin con số này.
    EXPECTED_VIDEOS = 611
PARTS_TIER = "window_records"
DATA_DIR = "data/vectors"

# ── ĐƯỜNG DỰ PHÒNG: chạm trần 5 tiếng thì sang Factory, mở lại thì quay về ──
#
# Trammay là PRIMARY và điều đó không đổi: cổng gửi ~6.000 request mỗi cửa sổ
# 5 giờ (6 tài khoản thượng nguồn × ~1.000), còn Factory bị cron ghì ở nhịp
# **1 phút cho mỗi automation**. Factory không thay được Trammay; nó chỉ lấp
# quãng NGỦ CHỜ mà trước đây worker nằm không.
#
# ⚠ Số dưới đây là GIẢ ĐỊNH, chưa đo. Trần thật cần ba phép: bao nhiêu cụm lọt
# trong MỘT prompt, bao nhiêu automation chạy song song được, và tài khoản
# Factory có trần ngày không. Đặt dè dặt cho tới khi có số.
BACKEND_DU_PHONG = "factory"      # "factory" | None (None = ngủ chờ như cũ)
FACTORY_SECRET = "API_KEY"        # tên secret Kaggle giữ khoá Factory
FACTORY_BASE = "https://api.factory.8090.dev/v2/external-api"
FACTORY_MODEL = "gemini-3.8-flash"
FACTORY_SONG_SONG = 2             # automation cùng lúc — CHƯA ĐO trần, đừng nâng bừa
FACTORY_CHO_GIAY = 300            # chờ tối đa một automation trả lời
FACTORY_NHIP_POLL = 15            # giây giữa hai lần đọc agent_conversations
FACTORY_HONG_TOI_DA = 5           # hỏng liên tiếp bấy nhiêu lần → bỏ dự phòng, ngủ chờ
NHIP_DO_TRAMMAY_PHUT = 10         # đang ở Factory thì bao lâu thăm dò Trammay một lần

RESEARCH_VOLUME = CAPTION_VOLUME  # nơi ĐỌC caption và GHI bản ghi

if RUNNER not in RUNNERS:
    raise ValueError(f"RUNNER phải thuộc {RUNNERS}")
if RUN_SCOPE not in {"PREFLIGHT", "SHARD", "AGGREGATE"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT, SHARD hoặc AGGREGATE")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != RUNNER:
    raise RuntimeError("SHARD bị khoá; đặt CONFIRM_SHARD đúng bằng RUNNER")
if RUN_SCOPE == "AGGREGATE" and not CONFIRM_AGGREGATE:
    raise RuntimeError("AGGREGATE bị khoá; đặt CONFIRM_AGGREGATE = True")
if DUONG_GOI not in {"chat", "batch"}:
    raise ValueError('DUONG_GOI phải là "chat" hoặc "batch"')
if not 1 <= BATCH_SIZE <= 8:
    raise ValueError("BATCH_SIZE phải từ 1 đến 8")
if BATCH_MAX_BYTES < 700_000 or BATCH_MAX_BYTES > 7_500_000:
    # Trần thân request của cổng là 8 MB (Đạt nâng 22-09; đo: 5 MB qua, 8 MB
    # trả 413). 7,5 MB là mức cao nhất còn chừa biên cho overhead JSON.
    #
    # ⚠ KÍCH THƯỚC ẢNH KHÁC NHAU RẤT NHIỀU giữa hai batch và giữa các nhóm nội
    # dung — đo 23-09 trên 208.491 keyframe batch 2 (zip / số khung, theo worker):
    #
    #     batch 1              178 KB/ảnh  -> base64 237 KB -> k=8 = 1,85 MiB
    #     batch 2 M tin tức    388 KB      -> 517 KB        -> k=8 = 4,04 MiB
    #     batch 2 S đua xe     393 KB      -> 524 KB        -> k=8 = 4,10 MiB
    #     batch 2 N giao thông 582 KB      -> 776 KB        -> k=8 = 6,06 MiB
    #
    # Ảnh giao thông gấp 3,3 lần batch 1: 1080p đầy chi tiết tần số cao (mặt
    # đường, nhiều vật nhỏ, nhiễu đêm) nên JPEG nén kém. Để 4,5 MB như trước là
    # batch nhóm N tự cắt xuống ~5 ảnh, và `BATCH_SIZE = 8` thành con số nói dối.
    #
    # 7,0 MB chừa ~1 MB biên dưới trần 8 MB cho prompt + khung JSON. `_chia_batch`
    # cắt theo byte nên nó TỰ điều tiết: ảnh nhẹ thì đủ 8, ảnh nặng thì ít hơn.
    raise ValueError("BATCH_MAX_BYTES phải nằm trong [700_000, 7_500_000]")
if DUONG_GOI == "batch" and BACKOFF_BASE < 20:
    # Chỉ ràng cho đường BATCH. Lùi ngắn hơn thời gian một batch là quay lại
    # đúng lúc suất đồng thời vẫn bận. Đo: nền 2s -> 8/18 batch chết ở 4 luồng;
    # nền 30s -> 0/18 ở 2 luồng. Đường `chat` thì request chỉ ~8s nên nền 2s là
    # đúng — ràng nó theo luật của batch là bắt nó chờ vô cớ.
    raise ValueError("BACKOFF_BASE phải ≥ 20 khi DUONG_GOI == 'batch'")
if DUONG_GOI == "chat" and MAX_WORKERS > 28:
    # 28 là mức đã chạy 6–9 giờ sạch đêm 21-09. 30 luồng thì cổng trả
    # `EOF occurred in violation of protocol` hàng loạt — đứt ở tầng TLS.
    raise ValueError("MAX_WORKERS > 28 ở đường chat: 30 luồng đã đo ra đứt TLS")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUNNER={RUNNER} · RUN_SCOPE={RUN_SCOPE} · model={MODEL_NAME} · "
      f"prompt={PROMPT_VARIANT} · batch={BATCH_SIZE} · max_bytes={BATCH_MAX_BYTES:,}")

In [ ]:
import subprocess, sys, os

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# Repo là PRIVATE — `git clone` trần sẽ hỏi mật khẩu rồi TREO notebook, không
# phải báo lỗi. `GIT_ASKPASS` trả token qua stdin, `GIT_TERMINAL_PROMPT=0` biến
# mọi lần hỏi còn sót thành lỗi ngay thay vì treo. Cùng cách nb33 đã dùng.
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; '
        '*) printf %s "$GITHUB_TOKEN" ;; esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": secrets.get_secret("GITHUB_TOKEN"),
                "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL,
                    str(REPO_DIR)], check=True, env=env)
sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["TRAMMAY_API_KEY"] = secrets.get_secret(TRAMMAY_SECRET)
for name in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"):
    os.environ[name] = secrets.get_secret(name)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modal"], check=True)

from mfe.contracts.ids import build_keyframe_id
from mfe.offline.bundle import BundleLayout
from mfe.offline.caption_prompt import PROMPT_VARIANTS
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.trammay import QuotaError, TrammayClient, TrammayError, available

from mfe.offline.caption_prompt import loc_khoi_ocr, tach_caption_ocr
from mfe.offline.shot_record_prompt import (
    PROMPT_BAN_GHI_SHOT,
    dung_dau_vao,
    loc_ao_giac_thoai,
)

PROMPT = PROMPT_BAN_GHI_SHOT
PROMPT_SUFFIX = ""    # KHÔNG dán `len180` — xem docstring `_va_prompt`

ok, vi_sao = available()
if not ok:
    raise RuntimeError(vi_sao)

# Chữ ký là `(volume_name, *, create_if_missing, version, dotenv_path)` — tên
# volume đi VỊ TRÍ, và KHÔNG có tham số `local_root`. Gọi `volume=...` ném
# TypeError ngay ở ô này.
RESEARCH = ModalVolumeTransport(RESEARCH_VOLUME, create_if_missing=False)
BTC = ModalVolumeTransport(BTC_VOLUME, create_if_missing=False)
BTC_LAYOUT = BundleLayout(BTC_BUNDLE)
CACHE_ANH.mkdir(parents=True, exist_ok=True)

# `backoff_base=30`, KHÔNG phải mặc định 2,0.
#
# Trần đồng thời của cổng đếm request ĐANG BAY, và một batch 8 ảnh chạy 23–47s.
# Lùi 2s/4s/8s là thử lại vào đúng lúc các suất vẫn chưa trống — cả bốn lần đều
# trượt. Đo 22-09 trên cùng 140 khung, 2 luồng: nền 2s -> 1/18 batch chết; nền
# 30s -> **0/18**, 140/140 caption.
client = TrammayClient.from_env(
    timeout=REQUEST_TIMEOUT, max_retries=MAX_RETRIES, backoff_base=BACKOFF_BASE
)
client = client.__class__(**{**client.__dict__, "model": MODEL_NAME})
print(f"prompt {PROMPT_VARIANT}: {len(PROMPT)} ký tự · batch={BATCH_SIZE} · client sẵn sàng")

## Chia việc — theo VIDEO, không theo keyframe

Chia theo video để mỗi worker tự đủ: một video là một đơn vị marker, và hai worker không bao giờ ghi cùng một file. Phép chia là **xác định** (băm tên video), nên chạy lại ở máy khác vẫn ra đúng cùng phần việc — không cần bảng phân công lưu ở đâu cả.

**Danh sách video đọc từ chính volume BTC**, không từ dataset Kaggle. Phép băm chia việc chạy trên danh sách đó, nên sáu worker **bắt buộc** thấy cùng một tập — mà đó chính là điều kiện để không video nào bị bỏ sót.


In [ ]:
import csv, hashlib, json

def _liet_ke_video() -> list:
    """Video ĐÃ CÓ caption — đó là đầu vào của tầng này, không phải ảnh."""
    ra = []
    for name in RESEARCH.listdir(CAPTION_DIR):
        ten = name.rsplit("/", 1)[-1]
        if ten.endswith(".json"):
            ra.append(ten[:-5])
    return sorted(ra)

VIDEOS = _liet_ke_video()
print(f"volume BTC: {len(VIDEOS)} video")
# KHÔNG ném khi thiếu: caption còn đang sinh nên danh sách lớn dần. Nhưng
# PHẢI in ra, vì chia việc tính trên chính danh sách này (§5).
if len(VIDEOS) != EXPECTED_VIDEOS:
    print(f"⚠ {len(VIDEOS)}/{EXPECTED_VIDEOS} video có caption — còn "
          f"{EXPECTED_VIDEOS - len(VIDEOS)} video CHƯA sinh xong.\n"
          "   Shard tính trên danh sách HIỆN TẠI; chạy lại sau khi caption\n"
          "   xong sẽ thấy thêm video mới — đó là hành vi ĐÚNG.")

def shard_cua(video_id: str) -> str:
    """Băm xác định -> tên worker. Đổi hàm này là đổi toàn bộ phân công."""
    h = hashlib.sha256(video_id.encode()).hexdigest()
    return RUNNERS[int(h[:8], 16) % len(RUNNERS)]

PHAN_CONG = {r: [v for v in VIDEOS if shard_cua(v) == r] for r in RUNNERS}
for r in RUNNERS:
    print(f"  {r}: {len(PHAN_CONG[r]):>3} video")
CUA_TOI = PHAN_CONG[RUNNER] if RUN_SCOPE != "PREFLIGHT" else PHAN_CONG[RUNNER][:1]
print(f"\n{RUNNER} xử lý {len(CUA_TOI)} video trong lượt này")

## Cửa sổ thời gian — `window_id` là `{video}#w0012`\n\nTiền tố `#w` tách hẳn cửa sổ khỏi **ba** định dạng `shot_id` đang tồn tại trong kho (`#s00` của hợp đồng, `#s0000` của gói audit TransNetV2, `_s0` giữ chỗ trong `clean-candidates.jsonl`). Không ai được phép lẫn một cửa sổ 15 giây với một cú máy 2,7 giây.\n

In [ ]:
import csv as _csv
import re

def pts_cua(video_id: str) -> dict:
    """`frame_idx -> pts_time`, đọc từ `map-keyframes`.

    KHÔNG suy `pts = frame_idx / fps` với fps đoán: batch 1 có 4 giá trị fps
    (một video chạy 26,44) và batch 2 có 68. CSV là chỗ duy nhất mang đúng con
    số BTC dùng để chấm (§2).
    """
    local = WORK_DIR / "maps" / f"{video_id}.csv"
    local.parent.mkdir(parents=True, exist_ok=True)
    if not local.is_file():
        BTC.get(BTC_LAYOUT.map_keyframes(video_id), local)
    ra = {}
    with local.open(encoding="utf-8") as fh:
        for r in _csv.DictReader(fh):
            try:
                ra.setdefault(int(r["frame_idx"]), float(r["pts_time"]))
            except (KeyError, ValueError, TypeError):
                continue
    return ra

def doan_thoai_cua(video_id: str) -> list:
    """`[{start, end, text}]` của faster-whisper, từ CHÍNH volume của batch này.

    ⚠ Volume có HAI hình dạng cho cùng một tầng: `stages.py` ghi
    `{"language": …, "segments": [...]}`, nhưng đếm 24-09 trên 60 video batch 1
    thì **62% là mảng trần** `[{...}]` và 38% là dict. Batch 2 thì 100% dict.
    Đỡ cả hai — chọn một là nửa batch 1 lặng lẽ mất phụ đề, `[speech]` biến mất
    khỏi prompt, và bản ghi vẫn sinh ra bình thường nên không ai thấy.

    Không có file / video câm ⇒ `[]`. Đếm 24-09: **toàn bộ 296 video nhóm N
    (camera giao thông) là video câm**, còn M 304/304 và S 11/11 đều có tiếng.
    """
    local = WORK_DIR / "subs" / f"{video_id}.json"
    local.parent.mkdir(parents=True, exist_ok=True)
    if not local.is_file():
        try:
            BTC.get(BTC_LAYOUT.subtitle_segments(video_id), local)
        except Exception:
            return []
    try:
        d = json.loads(local.read_text(encoding="utf-8"))
    except Exception:
        return []
    doan = d if isinstance(d, list) else d.get("segments", [])
    return [s for s in doan
            if isinstance(s, dict) and (s.get("text") or "").strip()]

def phu_de_cua(doan: list, tu: float, den: float) -> list:
    """Lời thoại CHẠM `[tu, den)` — cùng phép giao của `stages.py:495`.

    `end > tu and start < den` chứ không phải `start >= tu`: một câu bắt đầu
    trước cửa sổ và kết thúc trong nó vẫn là lời đang nói ở cửa sổ này.

    ⚠ Lọc ẢO GIÁC của Whisper trước khi trả. Đo 24-09 trên 11 video nhóm S:
    `"Hãy subscribe cho kênh Ghiền Mì Gõ..."` có ở 8/11 video, tới ×220 trong
    `S01-V009` (25,7% số dòng). Lọc theo SỐ LẦN LẶP là sai — ngưỡng ≥10 ăn luôn
    `Thành phố Hồ Chí Minh` (×51) và tên tay đua (`Christoph Janssen` ×15).
    """
    return loc_ao_giac_thoai(
        [(s["text"] or "").strip() for s in doan
         if s.get("end", -1) > tu and s.get("start", 1e18) < den
         and (s.get("text") or "").strip()])

_FRAME_RE = re.compile(r"#(\d{6})$")

def _thua(ds: list, toi_da: int) -> list:
    """Lấy mẫu THƯA, giữ khung đầu và khung cuối của cửa sổ."""
    if len(ds) <= toi_da:
        return ds
    buoc = (len(ds) - 1) / (toi_da - 1)
    return [ds[round(k * buoc)] for k in range(toi_da)]

def cua_so_cua(video_id: str, caps: dict) -> list:
    """`[{window_id, tu_giay, keyframes, captions}]` — chỉ cửa sổ CÓ khung.

    Khung nào không tra được `pts_time` thì BỎ, không gán bừa: gán sai cửa sổ
    là đặt một sự kiện vào sai mốc thời gian, và không phép kiểm nào nhìn ra.
    """
    pts = pts_cua(video_id)
    doan = doan_thoai_cua(video_id)
    theo_khung = []
    thieu_pts = 0
    for kid, text in caps.items():
        m = _FRAME_RE.search(kid)
        if not m:
            continue
        fi = int(m.group(1))
        if fi not in pts:
            thieu_pts += 1
            continue
        cap = tach_caption_ocr(text)[0] if CAPTION_THO else text
        if cap.strip():
            theo_khung.append((pts[fi], fi, kid, cap))
    if thieu_pts:
        print(f"  ⚠ {video_id}: {thieu_pts} khung không tra được pts_time, đã bỏ")
    theo_khung.sort()

    nhom = {}
    for t, fi, kid, cap in theo_khung:
        nhom.setdefault(int(t // CUA_SO_GIAY), []).append((fi, kid, cap))
    ra = []
    for k, ds in sorted(nhom.items()):
        ds = _thua(ds, MAX_KF_TRONG_CUA_SO)
        tu = k * CUA_SO_GIAY
        ra.append({"window_id": f"{video_id}#w{k:04d}",
                   "tu_giay": tu,
                   "keyframes": [kid for _, kid, _ in ds],
                   "captions": [c for _, _, c in ds],
                   # Thưa keyframe KHÔNG được thưa lời thoại: thoại cắt theo
                   # THỜI GIAN của cửa sổ, không theo khung nào còn lại.
                   "subtitles": phu_de_cua(doan, tu, tu + CUA_SO_GIAY)})
    return ra

def chia_cum(cs: list) -> list:
    """Cắt theo SỐ CAPTION, không theo số cửa sổ — độ dài prompt do caption quyết."""
    cum, hien, n = [], [], 0
    for c in cs:
        if hien and (len(hien) >= CUM_CUA_SO_TOI_DA
                     or n + len(c["captions"]) > CUM_CAPTION_TOI_DA):
            cum.append(hien)
            hien, n = [], 0
        hien.append(c)
        n += len(c["captions"])
    if hien:
        cum.append(hien)
    return cum

_thu_v = CUA_TOI[0]
_thu_caps = {}
try:
    _p = WORK_DIR / "caps" / f"{_thu_v}.json"
    _p.parent.mkdir(parents=True, exist_ok=True)
    RESEARCH.get(f"{CAPTION_DIR}/{_thu_v}.json", _p)
    _thu_caps = json.loads(_p.read_text(encoding="utf-8"))
except Exception as _e:
    print(f"⚠ chưa đọc được caption của {_thu_v}: {_e!r}")
if _thu_caps:
    _cs = cua_so_cua(_thu_v, _thu_caps)
    _cum = chia_cum(_cs)
    _co_th = sum(1 for c in _cs if c["subtitles"])
    print(f"{_thu_v}: {len(_thu_caps)} caption -> {len(_cs)} cửa sổ "
          f"{CUA_SO_GIAY:.0f}s -> {len(_cum)} cụm · "
          f"{sum(len(c['captions']) for c in _cs) / max(len(_cs), 1):.1f} khung/cửa sổ · "
          f"{_co_th}/{len(_cs)} cửa sổ có lời thoại")
    if _co_th == 0:
        print("  (video câm — đúng với nhóm N; khối [speech] sẽ không xuất hiện)")

In [ ]:
import json as _json
import threading
import time
import urllib.error
import urllib.request

# ───────────────────────────── ĐƯỜNG GỌI HAI TẦNG ─────────────────────────────
#
# Trammay PRIMARY, Factory dự phòng. Vì sao thứ tự đó và không đảo:
#
#   Trammay  ~6.000 request / cửa sổ 5 giờ (6 tài khoản thượng nguồn × ~1.000)
#   Factory  cron nhanh nhất 1 phút/automation ⇒ ~300 lượt / 5 giờ với 1 luồng
#
# Factory chậm hơn ~20 lần. Nó KHÔNG thay Trammay được — nó chỉ lấp quãng mà
# trước đây worker nằm không chờ hạn mức mở lại (đo 24-09: một worker dừng ở
# 4,95 giờ rồi ngủ suốt đêm). Lấp được bao nhiêu thì chưa đo; xem cảnh báo ở ô
# cấu hình.
#
# ⚠ Factory KHÔNG có endpoint `chat/completions` (tài liệu §18.1). Cách duy
# nhất gọi model là tạo một Automation có trigger cron, đợi nó chạy, đọc
# `agent_conversations`, rồi TẮT. Tắt là bắt buộc: `* * * * *` chạy mỗi phút
# cho tới khi bị tắt, và một automation bỏ quên là rác chạy ngầm trong project
# của Đạt.


class _BoDieuPhoi:
    """Giữ backend hiện hành và chuyển qua lại, an toàn với nhiều luồng.

    Một trạng thái dùng chung cho cả `MAX_WORKERS` luồng, nên mọi lần đọc/ghi
    đều qua khoá. Không dùng biến toàn cục rời: hai luồng cùng chạm trần sẽ tạo
    hai lần chuyển và in hai lần, còn tệ hơn là cùng thăm dò Trammay song song
    và đốt request vào việc thăm dò.
    """

    def __init__(self):
        self.backend = "trammay"
        self.khoa = threading.Lock()
        self.lan_do_cuoi = 0.0
        self.dem = {"trammay": 0, "factory": 0}
        self.so_lan_chuyen = 0
        self.factory_hong_lien_tiep = 0

    def hien_tai(self) -> str:
        with self.khoa:
            return self.backend

    def bao_het_quota(self) -> str:
        """Chạm trần: chuyển sang dự phòng. Trả backend sẽ dùng tiếp."""
        with self.khoa:
            if self.backend == "trammay":
                if not BACKEND_DU_PHONG:
                    return "ngu"
                self.backend = "factory"
                self.so_lan_chuyen += 1
                self.lan_do_cuoi = time.time()
                print(f"   ⛔ Trammay hết hạn mức → chuyển sang {BACKEND_DU_PHONG} "
                      f"(lần {self.so_lan_chuyen})", flush=True)
            return self.backend

    def den_luc_do_lai(self) -> bool:
        """Đúng MỘT luồng được đi thăm dò; các luồng khác cứ chạy Factory tiếp."""
        with self.khoa:
            if self.backend != "factory":
                return False
            if time.time() - self.lan_do_cuoi < NHIP_DO_TRAMMAY_PHUT * 60:
                return False
            self.lan_do_cuoi = time.time()
            return True

    def bao_trammay_mo(self) -> None:
        with self.khoa:
            if self.backend != "trammay":
                self.backend = "trammay"
                print("   ✅ Trammay mở lại → quay về primary", flush=True)

    def ghi_nhan(self, ten: str) -> None:
        with self.khoa:
            self.dem[ten] = self.dem.get(ten, 0) + 1
            if ten == "factory":
                self.factory_hong_lien_tiep = 0

    def factory_hong(self) -> bool:
        """Đếm hỏng LIÊN TIẾP. `True` = bỏ đường dự phòng, quay về ngủ chờ.

        Không có van này thì Factory chết (trần tài khoản, mạng, đổi API) sẽ làm
        MỌI cụm trượt trong khi vòng lặp vẫn chạy tiếp — đốt hết phiên Kaggle mà
        không sinh được gì, và Kaggle vẫn báo `Successfully ran`. Ngủ chờ tệ hơn
        chạy được, nhưng tốt hơn nhiều so với chạy mà không ra gì.
        """
        with self.khoa:
            self.factory_hong_lien_tiep += 1
            if self.factory_hong_lien_tiep < FACTORY_HONG_TOI_DA:
                return False
            self.backend = "trammay"      # để vòng ngoài thấy QuotaError mà ngủ
            self.factory_hong_lien_tiep = 0
            print(f"   ⚠ Factory hỏng {FACTORY_HONG_TOI_DA} lần liên tiếp → "
                  f"bỏ đường dự phòng, quay về NGỦ CHỜ hạn mức", flush=True)
            return True

    def tom_tat(self) -> str:
        with self.khoa:
            return (f"trammay {self.dem['trammay']} · factory {self.dem['factory']} "
                    f"· chuyển {self.so_lan_chuyen} lần")


DIEU_PHOI = _BoDieuPhoi()
_FACTORY_SUAT = threading.Semaphore(FACTORY_SONG_SONG)


def _factory_goi(duong: str, *, method: str = "GET", than=None) -> dict:
    req = urllib.request.Request(
        FACTORY_BASE + duong,
        data=_json.dumps(than).encode() if than is not None else None,
        headers={"X-API-Key": os.environ["FACTORY_API_KEY"],
                 "Content-Type": "application/json"},
        method=method,
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            return _json.loads(r.read().decode("utf-8")) if r.status != 204 else {}
    except urllib.error.HTTPError as e:
        # KHÔNG in header: nó mang API key. Chỉ in thân lỗi.
        raise RuntimeError(
            f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:200]}") from e


def _factory_chat(prompt: str) -> str:
    """Một lượt hỏi qua Factory. Ném nếu không lấy được câu trả lời.

    Automation LUÔN bị tắt trong `finally`, kể cả khi vỡ giữa chừng.
    """
    ten = f"mfe-cuaso-{RUNNER}-{int(time.time() * 1000)}"
    auto_id = None
    with _FACTORY_SUAT:
        try:
            d = _factory_goi("/automations", method="POST", than={
                "name": ten,
                "user_prompt": (
                    prompt
                    + "\n\nDo NOT create, edit or delete anything in this project. "
                      "Do NOT call any tool. Reply with the JSON object only, as "
                      "your final message."
                ),
                "triggers": [{"trigger_type": "cron",
                              "config": {"cron_expression": "* * * * *",
                                         "timezone": "UTC"}}],
                "model": FACTORY_MODEL,
                "model_configuration": {"thinking_level": "low"},
                # Agent có tool ghi vào project của Đạt. Chặn hết — nó chỉ cần trả lời.
                "tool_kit": {"permission_overrides": [
                    {"domain": "requirements", "restriction": "cannot_edit_or_create"},
                    {"domain": "blueprints", "restriction": "cannot_edit_or_create"},
                    {"domain": "work_orders", "restriction": "cannot_create"},
                    {"domain": "knowledge_base", "restriction": "cannot_edit_or_create"},
                ]},
            })
            auto_id = (d.get("data") or d).get("id")
            t0 = time.time()
            while time.time() - t0 < FACTORY_CHO_GIAY:
                time.sleep(FACTORY_NHIP_POLL)
                for h in _factory_goi("/agent_conversations").get("items", []):
                    au = h.get("automation") or {}
                    if au.get("automation_name") != ten:
                        continue
                    if au.get("run_status") not in {"success", "completed"}:
                        continue
                    ct = _factory_goi(f"/agent_conversations/{h['id']}")["data"]
                    tn = [m for m in ct.get("content", [])
                          if m.get("role") == "assistant"]
                    if tn and tn[-1].get("text"):
                        return tn[-1]["text"]
            raise RuntimeError(f"Factory không trả lời trong {FACTORY_CHO_GIAY}s")
        finally:
            if auto_id:
                try:
                    _factory_goi(f"/automations/{auto_id}", method="PATCH",
                                 than={"enabled": False})
                except Exception as e:
                    print(f"   ⚠ KHÔNG tắt được automation {auto_id}: {e} — "
                          f"tắt TAY ngay, nó chạy mỗi phút!", flush=True)


def goi_model(prompt: str) -> tuple:
    """`(text, backend)`. Tự chuyển Trammay ↔ Factory. Ném khi cả hai hỏng.

    Không bắt `TrammayError` ở đây: lỗi 5xx/timeout là chuyện của `_post` (đã
    có 4 lần thử với giãn cách), còn ở tầng này chỉ phân biệt ĐÚNG MỘT thứ —
    hết hạn mức hay chưa. Gộp hai loại lỗi là mất khả năng biết vì sao chậm.
    """
    if DIEU_PHOI.hien_tai() == "factory" and DIEU_PHOI.den_luc_do_lai():
        try:
            client.chat("ping", max_tokens=8)
            DIEU_PHOI.bao_trammay_mo()
        except QuotaError:
            pass                      # vẫn đóng, chạy Factory tiếp
        except TrammayError:
            pass                      # 5xx không nói gì về hạn mức — chưa biết

    if DIEU_PHOI.hien_tai() == "trammay":
        try:
            ra = client.chat(prompt, max_tokens=MAX_TOKENS_CHAT, temperature=0.0)
            DIEU_PHOI.ghi_nhan("trammay")
            return ra, "trammay"
        except QuotaError:
            if DIEU_PHOI.bao_het_quota() == "ngu":
                raise                 # không có dự phòng → để vòng ngoài ngủ chờ

    try:
        ra = _factory_chat(prompt)
    except Exception:
        if DIEU_PHOI.factory_hong():
            raise QuotaError("Factory hỏng liên tiếp — quay về ngủ chờ hạn mức") from None
        raise
    DIEU_PHOI.ghi_nhan("factory")
    return ra, "factory"


if BACKEND_DU_PHONG == "factory":
    os.environ["FACTORY_API_KEY"] = secrets.get_secret(FACTORY_SECRET)
    print(f"đường gọi: trammay (primary) → factory (dự phòng, "
          f"{FACTORY_SONG_SONG} automation song song, thăm dò lại mỗi "
          f"{NHIP_DO_TRAMMAY_PHUT} phút)")
else:
    print("đường gọi: trammay (primary), KHÔNG có dự phòng → chạm trần là ngủ chờ")

In [ ]:
import time
from datetime import datetime, timezone

def sha256_file(p) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for khoi in iter(lambda: fh.read(1 << 20), b""):
            h.update(khoi)
    return h.hexdigest()

def duong_remote(video_id: str) -> dict:
    return {
        "records": f"{OUTPUT_ROOT}/parts/{PARTS_TIER}/{video_id}.json",
        "marker": f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json",
        # Checkpoint ở thư mục KHÁC: "có file trong parts/" phải tiếp tục có
        # nghĩa là "video đã xong", vì AGGREGATE đọc đúng chỗ đó.
        "checkpoint": f"{OUTPUT_ROOT}/checkpoints/{RUNNER}/{video_id}.json",
    }

def marker_hop_le(video_id: str):
    """Marker là điểm chốt, không phải file — part đẩy dở vẫn tồn tại trên volume.

    `ModalVolumeTransport` KHÔNG có `.read()`; phải `get()` về rồi đọc.
    """
    paths = duong_remote(video_id)
    if not RESEARCH.exists(paths["marker"]):
        return None
    local = WORK_DIR / "marker-check" / f"{video_id}.json"
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        RESEARCH.get(paths["marker"], local)
        m = json.loads(local.read_text(encoding="utf-8"))
    except Exception:
        return None
    if m.get("status") != "COMPLETE" or m.get("runner") != RUNNER:
        return None
    if m.get("prompt_variant") != PROMPT_VARIANT or m.get("model") != MODEL_NAME:
        return None
    if m.get("cua_so_giay") != CUA_SO_GIAY:
        return None      # đổi độ dài cửa sổ là đổi đơn vị -> phải sinh lại
    if not RESEARCH.exists(paths["records"]):
        return None
    if m.get("records_sha256") != RESEARCH.sha256(paths["records"]):
        return None
    return m

def _tach_json(raw: str):
    """Model hay bọc JSON trong ```json … ```. Bóc ra, đừng chết vì cái vỏ."""
    s = raw.strip()
    if "```" in s:
        for k in s.split("```"):
            k = k.removeprefix("json").strip()
            if k.startswith("{"):
                s = k
                break
    i, j = s.find("{"), s.rfind("}")
    if i < 0 or j <= i:
        return None
    try:
        return json.loads(s[i:j + 1])
    except json.JSONDecodeError:
        return None

def mot_cum(cum, bo_nho_truoc):
    """Một cụm cửa sổ -> `(dict window_id->bản ghi, memory, lỗi|None)`.

    Đối chiếu `segment_id` trả về với danh sách gửi đi và VỨT cả cụm khi lệch:
    một bản ghi gắn nhầm cửa sổ là sai mốc thời gian, và không phép kiểm nội
    dung nào nhìn ra — cùng lớp bẫy với `chat_batch` kiểm tập id.
    """
    mong = [c["window_id"] for c in cum]
    goi = [{"segment_id": c["window_id"], "duration_seconds": CUA_SO_GIAY,
            "captions": c["captions"],
            "subtitles": c.get("subtitles") or []} for c in cum]
    try:
        raw, backend = goi_model(PROMPT + "\n\n" + dung_dau_vao(goi, bo_nho_truoc))
    except QuotaError:
        raise                      # không có dự phòng → vòng ngoài ngủ chờ
    except TrammayError as e:
        return {}, None, f"gọi hỏng: {str(e)[:120]}"
    except Exception as e:         # Factory hỏng: coi như cụm này trượt, đi tiếp
        return {}, None, f"factory hỏng: {type(e).__name__}: {str(e)[:100]}"
    d = _tach_json(raw)
    if not isinstance(d, dict) or not isinstance(d.get("segments"), list):
        return {}, None, "không parse được JSON"
    ra = {}
    for b in d["segments"]:
        sid = b.get("segment_id")
        if sid in mong and sid not in ra:
            # `onscreen_text` đi qua ĐÚNG bộ lọc đã dùng cho caption batch 2:
            # đo 24-09 thấy model vẫn nhặt mảnh logo `60 giây` vào đây.
            b["onscreen_text"] = [x for x in loc_khoi_ocr(
                "\n".join(b.get("onscreen_text") or [])).splitlines() if x]
            b["program"] = d.get("program")
            # `story` là bối cảnh CẤP CỤM (ai, ở đâu, vì sao) — phần lớn đến từ
            # phụ đề, thứ mà cụm sau sẽ KHÔNG còn nhìn thấy. Lưu kèm từng bản
            # ghi để sau này truy được một khẳng định trong `prose` về nguồn của
            # nó; `prose` đã dệt sẵn nội dung này nên chuỗi phẳng không cần nó.
            b["story"] = d.get("story")
            # Backend nào sinh ra bản ghi này. Hai đường dùng CÙNG prompt và
            # CÙNG model nhưng khác hạ tầng, nên nếu sau này đo thấy chất lượng
            # lệch thì đây là chỗ duy nhất truy ra được — đúng bài học
            # `prompt_suffix`: không truy được là không so được.
            b["backend"] = backend
            ra[sid] = b
    if set(ra) != set(mong):
        thieu = [x for x in mong if x not in ra]
        return {}, None, f"lệch segment_id (thiếu {thieu[:3]})"
    return ra, d.get("memory"), None

def xu_ly_video(video_id: str, publish: bool) -> dict:
    local = WORK_DIR / "parts" / video_id
    local.mkdir(parents=True, exist_ok=True)
    file_rec = local / "records.json"
    remote = duong_remote(video_id)
    if not file_rec.is_file() and publish and RESEARCH.exists(remote["checkpoint"]):
        try:
            RESEARCH.get(remote["checkpoint"], file_rec)
            if not isinstance(json.loads(file_rec.read_text(encoding="utf-8")), dict):
                file_rec.unlink(missing_ok=True)
        except Exception:
            file_rec.unlink(missing_ok=True)
    recs = json.loads(file_rec.read_text(encoding="utf-8")) if file_rec.is_file() else {}

    cap_local = WORK_DIR / "caps" / f"{video_id}.json"
    cap_local.parent.mkdir(parents=True, exist_ok=True)
    if not cap_local.is_file():
        RESEARCH.get(f"{CAPTION_DIR}/{video_id}.json", cap_local)
    caps = json.loads(cap_local.read_text(encoding="utf-8"))
    cs = cua_so_cua(video_id, caps)
    con_lai = [c for c in chia_cum(cs)
               if any(x["window_id"] not in recs for x in c)]

    t0, hong, bo_nho = time.time(), 0, None
    for k, cum in enumerate(con_lai, 1):
        phan, mem, loi = mot_cum(cum, bo_nho)
        if loi:
            hong += 1
            print(f"  cụm {k}/{len(con_lai)} hỏng: {loi}")
        else:
            recs.update(phan)
            bo_nho = mem      # hồi quy ở mức CỤM, không phải mức cửa sổ
        if k % 5 == 0:
            file_rec.write_text(json.dumps(recs, ensure_ascii=False), encoding="utf-8")
            if publish:
                RESEARCH.put(file_rec, remote["checkpoint"])
    file_rec.write_text(json.dumps(recs, ensure_ascii=False), encoding="utf-8")

    bao_cao = {"video_id": video_id, "caption": len(caps), "cua_so": len(cs),
               "ban_ghi": len(recs), "cum": len(con_lai), "hong": hong,
               "giay": round(time.time() - t0, 1)}
    if not publish:
        return bao_cao
    if not cs:
        # Video có caption nhưng không cửa sổ nào dựng được (thiếu pts_time).
        # KHÔNG ghi marker: `0 == 0` lọt qua phép kiểm PARTIAL và video biến mất
        # khỏi mọi báo cáo thiếu — đúng bẫy đã gặp với `M09_V026` ở nb35.
        bao_cao["status"] = "KHONG_CO_CUA_SO"
        return bao_cao
    if len(recs) != len(cs):
        bao_cao["status"] = "PARTIAL"
        return bao_cao

    RESEARCH.put(file_rec, remote["records"])
    if RESEARCH.sha256(remote["records"]) != sha256_file(file_rec):
        raise RuntimeError(f"{video_id}: records hash mismatch sau khi đẩy")
    marker = {"schema_version": 1, "status": "COMPLETE", "runner": RUNNER,
              "video_id": video_id, "windows": len(recs), "model": MODEL_NAME,
              "prompt_variant": PROMPT_VARIANT, "batch": BATCH,
              "cua_so_giay": CUA_SO_GIAY, "cum": len(con_lai),
              "records_sha256": sha256_file(file_rec),
              "heldout_accessed": False, "btc_mutated": False,
              "completed_at_utc": datetime.now(timezone.utc).isoformat()}
    tmp = local / "marker.json"
    tmp.write_text(json.dumps(marker, ensure_ascii=False, indent=1), encoding="utf-8")
    RESEARCH.put(tmp, remote["marker"])     # marker ghi SAU CÙNG
    bao_cao["status"] = "COMPLETE"
    return bao_cao

In [ ]:
def cho_het_quota() -> bool:
    """Ngủ tới khi cửa sổ hạn mức thượng nguồn mở lại. `True` = đi tiếp được.

    Vì sao NGỦ chứ không thoát: cửa sổ hạn mức là 5 tiếng còn phiên Kaggle sống
    12 tiếng, nên chờ hết cửa sổ rồi làm tiếp vẫn còn dư thời gian — trong khi
    thoát ra là phải có người ngồi canh để bấm chạy lại. Đo 24-09: một worker
    dừng lúc 4,95 giờ rồi nằm không suốt đêm.

    Vì sao THĂM DÒ chứ không ngủ đúng 5 tiếng: không ai biết cửa sổ bắt đầu lúc
    nào. Chạm trần ở phút thứ 10 của cửa sổ thì phải chờ 4h50; chạm ở phút 290
    thì chỉ chờ 10 phút. Ngủ cứng 5 tiếng là **luôn** chọn trường hợp xấu nhất.

    Thăm dò bằng một lượt gọi KHÔNG ẢNH (vài chục token, và quan trọng hơn là
    **một** request), nên 5 tiếng chờ tốn tối đa ~30 request trên 1.000 của cửa
    sổ. Lỗi KHÁC quota (5xx, timeout) không nói gì về hạn mức — coi như chưa
    biết và chờ tiếp, chứ đừng kết luận đã mở.

    Phiên có chết giữa lúc chờ cũng không mất gì: marker và checkpoint đã nằm
    trên volume, chạy lại là tiếp đúng chỗ dở.
    """
    t0 = time.time()
    tran = CHO_QUOTA_PHUT * 60
    while time.time() - t0 < tran:
        time.sleep(NHIP_THAM_DO_PHUT * 60)
        phut = (time.time() - t0) / 60
        try:
            client.chat("ping", max_tokens=8)
            print(f"   ✅ hạn mức đã mở lại sau {phut:.0f} phút chờ", flush=True)
            return True
        except QuotaError:
            print(f"   … vẫn hết hạn mức ({phut:.0f}/{CHO_QUOTA_PHUT} phút)", flush=True)
        except TrammayError as e:
            # Không phải tín hiệu về quota. Chờ tiếp thay vì đoán bừa.
            print(f"   … thăm dò lỗi khác ({phut:.0f} phút): {str(e)[:70]}", flush=True)
    return False

if RUN_SCOPE in {"PREFLIGHT", "SHARD"}:
    xong_roi = ([v for v in CUA_TOI if marker_hop_le(v)]
                if RUN_SCOPE == "SHARD" else [])
    con_lai = [v for v in CUA_TOI if v not in set(xong_roi)]
    print(f"{len(xong_roi)} video đã COMPLETE · còn {len(con_lai)}")

    bao_caos, that_bai = [], []
    dung_han = False
    for i, video_id in enumerate(con_lai, 1):
        if dung_han:
            break
        # Vòng trong CHỈ lặp khi chờ được hạn mức mở lại — để làm tiếp ĐÚNG
        # video đang dở, không bỏ nó lại rồi nhảy sang video sau.
        while True:
            try:
                r = xu_ly_video(video_id, publish=RUN_SCOPE == "SHARD")
                bao_caos.append(r)
                print(f"[{i}/{len(con_lai)}] {video_id}: {r['caption']}/{r['khung']} "
                      f"· hỏng {r['hong']} · {r['giay']}s · {r.get('status', 'PREFLIGHT')}")
            except QuotaError as e:
                # KHÔNG quét tiếp danh sách khi hạn mức đã cạn: làm thế chỉ tạo
                # ra một loạt video PARTIAL và đốt hết phiên (đo 23-09: 1.419,9s
                # cho MỘT video mà thu về 0 caption). Ngủ chờ cửa sổ reset rồi
                # làm tiếp chính video này.
                print(f"\n⛔ HẾT HẠN MỨC ở video {i}/{len(con_lai)} ({video_id})")
                print(f"   {str(e)[:200]}")
                print(f"   Chờ cửa sổ hạn mức reset, thăm dò mỗi "
                      f"{NHIP_THAM_DO_PHUT} phút…", flush=True)
                if cho_het_quota():
                    continue          # làm lại CHÍNH video này
                print(f"   Chờ quá {CHO_QUOTA_PHUT} phút mà chưa mở. DỪNG lượt chạy;")
                print("   phần đã xong giữ nguyên marker, chạy lại là tiếp chỗ dở.")
                dung_han = True
            except Exception as e:
                that_bai.append({"video_id": video_id, "loi": repr(e)})
                print(f"[{i}/{len(con_lai)}] {video_id}: THẤT BẠI {e!r}")
            break

    tong_hong = sum(r["hong"] for r in bao_caos)
    tong_cap = sum(r["caption"] for r in bao_caos)
    print(f"\nđường gọi: {DIEU_PHOI.tom_tat()}")
    print(f"\nTổng: {tong_cap:,} caption · {tong_hong} khung hỏng "
          f"({tong_hong / max(1, tong_cap + tong_hong) * 100:.1f}%)")
    if tong_hong / max(1, tong_cap + tong_hong) > 0.02:
        print("⚠ >2% hỏng — xem dòng `hỏng` ở trên để biết vì sao:\n"
              "   429 quota -> DỪNG, đợi cửa sổ hạn mức reset (5 tiếng)\n"
              "   HTTP 413  -> hạ BATCH_MAX_BYTES\n"
              "   max_tokens -> nới MAX_TOKENS\n"
              "   EOF/TLS    -> hạ MAX_WORKERS")

In [ ]:
if RUN_SCOPE == "AGGREGATE":
    if not CONFIRM_AGGREGATE:
        raise RuntimeError("đặt CONFIRM_AGGREGATE = True sau khi cả 6 shard COMPLETE")
    xong = [v for v in VIDEOS if marker_hop_le(v)]
    print(f"{len(xong)}/{len(VIDEOS)} video có marker hợp lệ")
    if len(xong) != len(VIDEOS):
        thieu = [v for v in VIDEOS if v not in set(xong)]
        raise RuntimeError(f"còn {len(thieu)} video chưa COMPLETE: {thieu[:5]}")

    gop = {}
    for i, video_id in enumerate(xong, 1):
        dich = WORK_DIR / "gop" / f"{video_id}.json"
        dich.parent.mkdir(parents=True, exist_ok=True)
        RESEARCH.get(f"{OUTPUT_ROOT}/parts/{PARTS_TIER}/{video_id}.json", dich)
        gop.update(json.loads(dich.read_text(encoding="utf-8")))
        if i % 100 == 0:
            print(f"  gộp {i}/{len(xong)}")
    print(f"tổng {len(gop):,} bản ghi cửa sổ")

    # Hai artifact, hai mục đích:
    #   `window_records.json` giữ NGUYÊN cấu trúc, cho tầng cửa sổ + soi tay;
    #   `window_text.json`    là chuỗi phẳng cho BM25/BGE-M3.
    #
    # Chuỗi phẳng = `prose` + `onscreen_text` ĐÃ LỌC (Lead chốt 24-09). Ba lý do:
    #
    #   - `prose` là CÂU THẬT, mà `bge_m3` là bộ nhúng câu. Chuỗi nối kiểu
    #     "X . Y . Z" không phải câu và nhúng kém hơn.
    #   - `prose` NUỐT được quan hệ liên cửa sổ vào nội dung: `link_to_previous`
    #     ở dạng nhãn (`"same subject closer"`) là stopword vô dụng cho BM25,
    #     nhưng "In a closer view of the scene, the man crouches…" thì khớp
    #     được với đề. Nên `link_to_previous` vẫn KHÔNG vào chuỗi ở dạng thô.
    #   - `entities` bị BỎ vì `prose` đã nhắc chúng; giữ cả hai là đếm hai lần.
    #
    # `onscreen_text` thì GIỮ dù prose đã trích: nó là bản ĐÃ QUA `loc_khoi_ocr`,
    # tức đã gỡ mảnh logo — thứ mà bản nằm trong prose không lọc được.
    phang, rong = {}, 0
    for wid, b in gop.items():
        # Rơi về `event` nếu model quên `prose`: thà ngắn còn hơn rỗng.
        phan = [b.get("prose") or b.get("event") or ""]
        phan += list(b.get("onscreen_text") or [])
        t = " . ".join(x for x in phan if x).strip()
        if t:
            phang[wid] = t
        else:
            rong += 1
    print(f"phẳng: {len(phang):,} chuỗi · {rong} bản ghi rỗng")

    ra = WORK_DIR / "window_records.json"
    ra.write_text(json.dumps(gop, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra, f"{OUTPUT_ROOT}/{DATA_DIR}/window_records.json")
    ra2 = WORK_DIR / "window_text.json"
    ra2.write_text(json.dumps(phang, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra2, f"{OUTPUT_ROOT}/{DATA_DIR}/window_text.json")
    print("đã đẩy window_records.json + window_text.json")

## Sau khi AGGREGATE\n\n`window_records.json` giữ nguyên cấu trúc; `window_text.json` là chuỗi phẳng cho BM25/BGE-M3.\n\n⚠ **Arm cửa sổ phát ra `keyframe_id`, không phát `window_id`**: BTC chấm `frame_idx`, nên truy hồi ở cấp cửa sổ rồi nở xuống khung con. Nhờ vậy `weighted_fusion` không phải sửa dòng nào.\n\n⚠ **Chốt 4 trước khi vào kho**: lấy 50 cửa sổ, người đọc đối chiếu `event` và `state_change` với video. Đo 24-09 thấy model bịa chuyển động (`A person walks along…` khi caption không nhắc ai đang đi) — ép có động từ thì nó sẽ dựng ra hành động.\n